## Functions, Parameter Mechanics & Call Stacks

Understanding how Python functions construct call frames and evaluate arguments is critical for building predictable interfaces and preventing memory leaks.

1. Function Parameter Kinds (PEP 570 & PEP 3102)
Python supports five distinct parameter kinds in a single signature, ordered strictly from left to right:

def func(pos_only, /, standard, *args, kw_only, **kwargs):
    pass
Positional-Only (/): Arguments before / cannot be passed by keyword (e.g., func(10) works, func(pos_only=10) raises TypeError).

Standard (Positional or Keyword): Can be passed either positionally or by name.

Variable Positional (*args): Collects extra positional arguments into a tuple.

Keyword-Only (* or after *args): Arguments after * or *args must be passed by keyword name (e.g., kw_only=True).

Arbitrary Keywords (**kwargs): Collects extra keyword arguments into a dict.

2. The Mutable Default Argument Bug Under the Hood
In Python, default parameter values are evaluated once when the function is defined (def), NOT every time the function is called.

Python
# DANGEROUS BUG:
def append_tensor(tensor_id, registry=[]):
    registry.append(tensor_id)
    return registry
At module load time, Python compiles the function into a PyFunctionObject.

The list [] is allocated in heap memory and stored directly inside the function's internal metadata: append_tensor.__defaults__.

Every subsequent call to append_tensor() mutates that single shared list in __defaults__.

Python
# The Idiomatic Fix:
def append_tensor(tensor_id, registry=None):
    if registry is None:
        registry = []
    registry.append(tensor_id)
    return registry

In [1]:
def allocate_gpu_buffer(
    device_id,
    byte_size,
    /,
    alignment=256,
    *extra_flags,
    pinned: bool = False,
    stream_id: int | None = None,
    **metadata
):
    return {
        "device_id": device_id,
        "byte_size": byte_size,
        "alignment": alignment,
        "extra_flags": extra_flags,
        "pinned": pinned,
        "stream_id": stream_id,
        "metadata": metadata,
    }

Absolutely. The easiest way to understand *args and **kwargs is:

*args → collects extra positional arguments
**kwargs → collects extra keyword arguments

In [2]:
print(allocate_gpu_buffer(0, 4096))

print(allocate_gpu_buffer(
    0,
    4096,
    512,
    b"READ",
    b"WRITE",
    pinned=True,
    stream_id=3,
    cache_policy="fast",
    owner="renderer",
))

{'device_id': 0, 'byte_size': 4096, 'alignment': 256, 'extra_flags': (), 'pinned': False, 'stream_id': None, 'metadata': {}}
{'device_id': 0, 'byte_size': 4096, 'alignment': 512, 'extra_flags': (b'READ', b'WRITE'), 'pinned': True, 'stream_id': 3, 'metadata': {'cache_policy': 'fast', 'owner': 'renderer'}}


In [3]:
def add_all(*args):
    total = 0

    for number in args:
        total += number

    return total

print(add_all(10, 20))
print(add_all(10, 20, 30, 40))

30
100


In [4]:
def show_info(**kwargs):
    print(kwargs)

show_info(name="Alice", age=25, city="Mysore")

{'name': 'Alice', 'age': 25, 'city': 'Mysore'}


In [9]:
def buggy_accumulator(val,cache=[]):
    cache.append(val)
    
buggy_accumulator(10)
print(buggy_accumulator.__defaults__)

buggy_accumulator(30)
print(buggy_accumulator.__defaults__)

buggy_accumulator(20)
print(buggy_accumulator.__defaults__)


([10],)
([10, 30],)
([10, 30, 20],)


In [11]:
def fixed_accumulator(val,cache=None):
    if cache==None:
        cache=[]
    cache.append(val)

In [12]:
fixed_accumulator(10)
print(fixed_accumulator.__defaults__)

fixed_accumulator(30)
print(fixed_accumulator.__defaults__)

fixed_accumulator(20)
print(fixed_accumulator.__defaults__)

(None,)
(None,)
(None,)
